In [0]:
# spark.conf.set("spark.checkpoint.dir", "/Volumes/databricks_practice/default/bronze/checkpoints")

In [0]:
# Country_code, Currency, Gender Segment tem nulos

In [0]:
import sys
sys.path.append("/Workspace/Shared/databricks-practice/utils/")

import pyspark.sql.functions as F

from functions import *

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS silver

In [0]:
bronze_table = "databricks_practice.bronze.bronze_table"
silver_table = "databricks_practice.silver.silver_table"
partition_col = "dt_ingestion_partition"


In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricks_practice.silver.silver_table (
  price_local DECIMAL(10, 2),
  sale_price_local DECIMAL(10, 2),
  discount_pct DECIMAL(10, 2),
  size_count NUMERIC,
  available_market BOOLEAN,
  available_size_count NUMERIC,
  snapshot_date DATE,
  product_id STRING,
  product_name STRING,
  model_number STRING,
  style_color STRING,
  brand_name STRING,
  category STRING,
  subcategory STRING,
  color_name STRING,
  sport_tags STRING,
  product_url STRING,
  canonical_url STRING,
  image_url STRING,
  sku STRING,
  catalog_sku_id STRING,
  stock_keeping_unit_id STRING,
  country_code STRING,
  currency STRING,
  dt_ingestion_partition DATE,
  update_ts TIMESTAMP
) USING DELTA;




In [0]:
df_new = load_bronze_new_partitions(spark, bronze_table, silver_table, partition_col)

In [0]:
df_silver = clean_mercado(df_new)

In [0]:
df_silver_tratado = df_silver \
   .withColumn("price_local", F.col("price_local").try_cast("decimal(10,2)")) \
   .withColumn("sale_price_local", F.col("sale_price_local").try_cast("decimal(10,2)")) \
   .withColumn("discount_pct", F.col("discount_pct").try_cast("decimal(10,2)")) \
   .withColumn("size_count", F.col("size_count").try_cast("double")) \
   .withColumn("available_market", F.col("available_market").try_cast("boolean")) \
   .withColumn("available_size_count", F.col("available_size_count").try_cast("double")) \
   .withColumn("snapshot_date", F.col("snapshot_date").try_cast("date")) \
   .withColumn("product_id", F.col("product_id").try_cast("string")) \
   .withColumn("product_name", F.col("product_name").try_cast("string")) \
   .withColumn("model_number", F.col("model_number").try_cast("string")) \
   .withColumn("style_color", F.col("style_color").try_cast("string")) \
   .withColumn("brand_name", F.col("brand_name").try_cast("string")) \
   .withColumn("category", F.col("category").try_cast("string")) \
   .withColumn("subcategory", F.col("subcategory").try_cast("string")) \
   .withColumn("color_name", F.col("color_name").try_cast("string")) \
   .withColumn("sport_tags", F.col("sport_tags").try_cast("string")) \
   .withColumn("product_url", F.col("product_url").try_cast("string")) \
   .withColumn("canonical_url", F.col("canonical_url").try_cast("string")) \
   .withColumn("image_url", F.col("image_url").try_cast("string")) \
   .withColumn("sku", F.col("sku").try_cast("string")) \
   .withColumn("catalog_sku_id", F.col("catalog_sku_id").try_cast("string")) \
   .withColumn("stock_keeping_unit_id", F.col("stock_keeping_unit_id").try_cast("string")) \
   .withColumn("country_code", F.col("country_code").try_cast("string")) \
   .withColumn("currency", F.col("currency").try_cast("string")) \
   .withColumn("dt_ingestion_partition", F.col("dt_ingestion_partition").try_cast("date")) \
   .withColumn("update_ts", F.current_timestamp())

In [0]:
display(df_silver_tratado)

In [0]:
df_silver_tratado = df_silver_tratado \
   .withColumn("chave", F.concat_ws("|",  F.col("product_id"),  F.col("country_code"), F.col("sku"), F.col("size_conversion_id"), F.col("snapshot_date")))


In [0]:
df_silver_tratado.count()

In [0]:
display(df_silver_tratado.groupby("chave").count())

In [0]:
df_silver_deduplicado = df_silver_tratado.dropDuplicates(["chave"])
df_silver_deduplicado.count()

In [0]:
df_silver_deduplicado.filter(
    F.col("chave").isNull()
).count()

In [0]:
df_silver_tratado.createOrReplaceTempView("df_silver_tratado")

display(spark.sql("""
SELECT
  product_id,
  sku,
  country_code,
  size_conversion_id,
  snapshot_date,
  COUNT(*) as qtd
FROM df_silver_tratado
GROUP BY
  product_id,
  sku,
  country_code,
  size_conversion_id,
  snapshot_date
HAVING COUNT(*) > 1
"""))

In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricks_practice.silver.silver_table (
  price_local DECIMAL(10, 2),
  sale_price_local DECIMAL(10, 2),
  discount_pct DECIMAL(10, 2),
  size_count DOUBLE,
  available_market BOOLEAN,
  available_size_count DOUBLE,
  snapshot_date DATE,
  product_id STRING,
  product_name STRING,
  model_number STRING,
  style_color STRING,
  brand_name STRING,
  category STRING,
  subcategory STRING,
  color_name STRING,
  sport_tags STRING,
  product_url STRING,
  canonical_url STRING,
  image_url STRING,
  sku STRING,
  catalog_sku_id STRING,
  stock_keeping_unit_id STRING,
  country_code STRING,
  currency STRING,
  dt_ingestion_partition DATE,
  update_ts TIMESTAMP
) USING DELTA;



In [0]:
df_silver_deduplicado = df_silver_deduplicado.drop("chave")

df_silver_deduplicado.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("databricks_practice.silver.silver_table")

In [0]:
%sql
SELECT * FROM databricks_practice.silver.silver_table

In [0]:
# checkpointed_df = df_global.checkpoint()